In [1]:
import os
import smtplib
import html
import requests
from datetime import date
from email.mime.text import MIMEText
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from easy_equities_client.clients import EasyEquitiesClient
from functions import buy_etf, get_free_cash, load_allocations, send_email

### Load variables

In [2]:
# Load .env file
load_dotenv()
today = date.today().strftime("%Y-%m-%d")

# Access variables
USERNAME = os.getenv("EE_USERNAME")
PASSWORD = os.getenv("EE_PASSWORD")
account_id = os.getenv("ZAR_TRADING_ACCOUNT_ID")
MIN_TRADE = float(os.getenv("MIN_TRADE"))
EMAIL_SENDER = os.getenv("EMAIL_SENDER")
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD")
EMAIL_RECEIVER = os.getenv("EMAIL_RECEIVER")

### Connect/Login to Easy Equities

In [3]:
client = EasyEquitiesClient()
client.login(username=USERNAME, password=PASSWORD)

True

### Determine free cash available to invest

In [4]:
# get free cash allocations on selected account
free_cash = get_free_cash(client, account_id)

### Determine allocation ammounts

In [5]:
# get the ticker, amount allocations
allocation_amounts = load_allocations(free_cash)

### Place buy orders

In [6]:
# iterate through all tickers  and execute buy trades
results = []
for ticker, data in allocation_amounts.items():
    amount = data["amount"]
    contract_code = data["contract_code"]
    ticker_name = data["ticker_name"]
    try:
        if amount < MIN_TRADE:
            results.append(f"SKIP {ticker_name}: Amount too small (R{amount:.2f})")
            continue

        # place buy order
        response = buy_etf(client, account_id, contract_code, amount)
        if response.status_code == 200:
            # Add results
            results.append(f"BUY {ticker_name}: R{amount:.2f} ✅")
        else:
            results.append(f"Failed to place order {ticker_name}: {response.text}")
    except Exception as e:
        results.append(f"Failed to place buy order for {ticker_name}:{e}")

### Send email with result of trade

In [7]:
# send main
send_email(results, today, EMAIL_SENDER, EMAIL_RECEIVER, EMAIL_PASSWORD)